[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/channel_extraction/sierra_de_gador_chi_analysis.ipynb)

# Basins, channel profiles and chi analysis in the Sierra de Gádor, Spain

The Sierra de Gádor is a limestone and dolomite range in Almería, southeast Spain, that rises to more than 2200 m only about 15 km from the Mediterranean coast. In this notebook we look at four river basins that drain the range, two to the south and two to the north, and ask how steep their channels are.

In this notebook you will:

1. Download a 30 m digital elevation model (DEM) with `lsdtt-fetch-raster`.
2. Make a hillshade with `lsdtt-raster-preprocessing`.
3. Extract the channel network, with stream orders, with `lsdtt-channel-extraction`.
4. Extract four basins from outlet points and run a **chi analysis** with `lsdtt-chi-analysis`.
5. Make four figures: a basin map, channel long profiles, chi plots with fitted segments, and the channel network draped over the hillshade.

The `lsdtt3` programs are command-line tools controlled by small text **parameter files** (`key: value` lines). We write these files from Python and then run the programs with `!`.

## Setup

### First set up condacolab

This step takes around 2 minutes. **`condacolab.install()` restarts the Colab runtime.** You will see a message that the session crashed: this is expected. Wait for it to reconnect, then carry on with the next cell.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

Now install `pygmt` (GMT is the Generic Mapping Tools). This is the slowest step and takes around a minute.

In [ ]:
!mamba install pygmt

Now get `lsdviztools3`, the plotting package.

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdviztools3-0.1.0-py3-none-any.whl

In [ ]:
!pip install "lsdviztools3-0.1.0-py3-none-any.whl[charts]"

Next, download and unpack the `lsdtt3` command-line programs.

In [ ]:
!wget https://www.geos.ed.ac.uk/~smudd/lsdtt_packages/lsdtt3-backend-linux-x86_64-core-v0.5.2.tar.gz

In [ ]:
!tar -xzf lsdtt3-backend-linux-x86_64-core-v0.5.2.tar.gz

Tell the system where to find the `lsdtt3` programs and their libraries. If your runtime restarts later, run this cell again.

In [ ]:
import os

root = "/content/lsdtt3-backend-linux-x86_64-core-v0.5.2"
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's older proj.db breaks rasterio and pyproj.

Check that it works: this prints the version of `lsdtt-chi-analysis`.

In [ ]:
!lsdtt-chi-analysis -v

## Step 1: Download a DEM

`lsdtt-fetch-raster` downloads a DEM for a box given in longitude and latitude (WGS84 degrees):

* `dem_source: cop30_aws`: the Copernicus GLO-30 DEM (about 30 m resolution), served from AWS. No account or API key is needed.
* `target_epsg: 32630`: reproject to **UTM zone 30N**, a coordinate system in metres. Flow routing and chi need distances in metres, not degrees.
* `grid_spacing: 30`: a 30 m output grid.

**A note on the box.** The box we started from ran from longitude −2.837 to −2.545 and latitude 36.788 to 37.008. But the outlet of our second basin is at longitude −2.861, which is *west* of that box, and its basin extends further west still (to about −2.875). So we moved the west edge out to **−2.89**. Always check that your outlets, and the whole of each basin upstream of them, are inside the DEM: `lsdtt-chi-analysis` only analyses **complete** basins, and a basin cut by the DEM edge gives a truncated, wrong chi profile.

The output is called `<write_fname>_DEM.tif`, so here `gador_DEM.tif` (about 1000 by 800 pixels).

In [ ]:
# Write the parameter file for lsdtt-fetch-raster
with open("gador_fetch.param", "w") as f:
    f.write("write_fname: gador\n")
    f.write("west: -2.89\n")                  # moved west from -2.837 to include basin 2
    f.write("east: -2.5452957146219943\n")
    f.write("south: 36.788063547917716\n")
    f.write("north: 37.00827540674705\n")
    f.write("dem_source: cop30_aws\n")
    f.write("target_epsg: 32630\n")
    f.write("grid_spacing: 30\n")

# Run it. The first argument is the directory holding the parameter file.
!lsdtt-fetch-raster ./ gador_fetch.param

## Step 2: Make a hillshade

`lsdtt-raster-preprocessing` fills pits in the DEM (`write_fill_raster`) and computes a hillshade from the filled DEM (`write_hillshade_raster`), lit from the northwest (`hillshade_azimuth: 315`) at 45 degrees above the horizon (`hillshade_altitude: 45`).

Outputs: `gador_fill.tif` and `gador_hillshade.tif`.

In [ ]:
with open("gador_hillshade.param", "w") as f:
    f.write("read_fname: gador_DEM.tif\n")
    f.write("write_fname: gador\n")
    f.write("write_fill_raster: true\n")
    f.write("write_hillshade_raster: true\n")
    f.write("hillshade_azimuth: 315\n")
    f.write("hillshade_altitude: 45\n")

!lsdtt-raster-preprocessing ./ gador_hillshade.param

## Step 3: Extract the channel network

`lsdtt-channel-extraction` routes flow with the D8 method (each pixel drains to its steepest downhill neighbour) and starts a channel wherever the number of upslope pixels reaches `threshold_contributing_pixels`. We use 200 pixels: with 30 m pixels (900 m² each) channels begin at a drainage area of 200 × 900 m² = 0.18 km². We use the same threshold in the chi analysis below so the two networks match.

Output: `gador_channel_network_lines.fgb`, one line per channel link with a `stream_order` (Strahler order) attribute.

In [ ]:
with open("gador_channels.param", "w") as f:
    f.write("read_fname: gador_DEM.tif\n")
    f.write("write_fname: gador\n")
    f.write("source_extraction_algorithm_choice: threshold\n")
    f.write("threshold_contributing_pixels: 200\n")
    f.write("write_channel_network_lines: true\n")

!lsdtt-channel-extraction ./ gador_channels.param

## Step 4: Basins and chi analysis

### What is chi?

Rivers that erode at the same rate get less steep downstream, because they carry more water. The **stream power law** says that, in steady state, channel slope $S$ and drainage area $A$ are related by

$$S = k_{sn} A^{-\theta}, \qquad \theta = m/n .$$

* $\theta = m/n$ is the **concavity index**: how quickly the channel flattens downstream. Many rivers have values between about 0.3 and 0.6; we use the common default **m/n = 0.45** (`m_over_n: 0.45`). You can test other values (see *Things to try*).
* $k_{sn}$ is the **normalised channel steepness index**. Once the effect of drainage area is removed, steeper channels (higher $k_{sn}$) usually mean faster uplift and erosion, or harder rock.

**Chi** ($\chi$) is a way to measure $k_{sn}$ from elevation instead of from noisy slopes. It is distance along the channel, from the outlet upstream, with each step weighted by $(A_0/A)^{m/n}$:

$$\chi = \int_{0}^{x} \left(\frac{A_0}{A(x')}\right)^{m/n} dx' .$$

If you plot elevation against $\chi$ (a **chi plot**), a channel in steady state is a straight line and its **gradient is $k_{sn}$** (with $A_0 = 1$ m², the `lsdtt3` default). Changes in gradient (kinks) show where $k_{sn}$ changes, for example at knickpoints or rock-type boundaries. If m/n is right, tributaries also plot on top of the main stem.

### The basins

We give `lsdtt-chi-analysis` a CSV of outlet points with `latitude` and `longitude` columns. The points we pick on a map are rarely exactly on the D8 channel, so we **snap** them: `basin_outlet_snapping_method: downstream` follows the flow path downstream from each point until it reaches a pixel with at least `basin_outlet_contributing_pixels_threshold` (here 200) upslope pixels, i.e. a channel pixel.

The basins are numbered from **0** in the order of the CSV rows: basin 0 is point 1, basin 1 is point 2, and so on.

In [ ]:
import pandas as pd

outlets = pd.DataFrame({
    "point":     [1, 2, 3, 4],
    "latitude":  [36.81287409584969, 36.836076686552246, 36.99417107455423, 36.990640095439055],
    "longitude": [-2.703219498076909, -2.860643532536022, -2.726459269311634, -2.674669372117472],
})
outlets.to_csv("gador_outlets.csv", index=False)
outlets

### Run `lsdtt-chi-analysis`

The main parameters:

* `threshold_contributing_pixels: 200`: the channel network used for chi (same as Step 3).
* `basin_outlet_fname`, `basin_outlet_snapping_method`, `basin_outlet_contributing_pixels_threshold`: the outlets and how to snap them (above).
* `n_largest_basins: 0`: keep all of our basins. (The default, 5, keeps only the 5 biggest *complete* basins.)
* `m_over_n: 0.45`: the concavity index.
* `write_chi_data_maps: true` with `chi_data_maps_write_vector_format: csv`: write every channel pixel with its chi, elevation, drainage area and flow distance (`gador_chi_data_maps.csv`), the snapped outlets (`gador_chi_basin_outlets.csv`) and the basin outlines (`gador_chi_basin_polygons.fgb`).
* `fit_chi_networks: true`: fit each chi profile with straight-line **segments**. A segment is split whenever the data move more than `fit_critical_divergence_metres` (here 10 m) away from a straight line. Each segment's gradient is its $k_{sn}$. The fit is written to `gador_chi_fit_nodes_mn_0p450.csv` (every pixel with observed and fitted elevation) and `gador_chi_fit_segments_mn_0p450.csv` (one row per segment with its `k_sn`). The `mn_0p450` part of the name is m/n = 0.450.
* `write_chi_fit_segment_lines: true`: the same segments as map lines, `gador_chi_fit_segment_lines_mn_0p450.fgb`, so we can map $k_{sn}$.

In the printed output, look for the line **`Complete basins: 4 / 4`**. If it says fewer than 4, a basin touches the edge of the DEM or a nodata area: make the box in Step 1 bigger.

In [ ]:
with open("gador_chi.param", "w") as f:
    f.write("read_fname: gador_DEM.tif\n")
    f.write("write_fname: gador\n")
    f.write("threshold_contributing_pixels: 200\n")
    f.write("basin_outlet_fname: gador_outlets.csv\n")
    f.write("basin_outlet_snapping_method: downstream\n")
    f.write("basin_outlet_contributing_pixels_threshold: 200\n")
    f.write("n_largest_basins: 0\n")
    f.write("m_over_n: 0.45\n")
    f.write("write_chi_data_maps: true\n")
    f.write("chi_data_maps_write_vector_format: csv\n")
    f.write("fit_chi_networks: true\n")
    f.write("fit_critical_divergence_metres: 10\n")
    f.write("write_fit_tables: true\n")
    f.write("write_chi_fit_segment_lines: true\n")

!lsdtt-chi-analysis ./ gador_chi.param

Let's look at the files we have made.

In [ ]:
!ls -lh gador_*

## Figure 1: Basin map

We load the basin outlines, check that none of them touches the edge of the DEM, and see how far each outlet point was moved by snapping. Then we map them with `lsdviztools3`:

* `render_hillshade` draws the hillshade (`shade=False` because it is already a hillshade).
* `render_basins` draws the basins semi-transparently, labelled with their number.
* `render_points` draws the points we asked for (red crosses) and the snapped outlets (black circles).

In [ ]:
from dataclasses import replace

import geopandas as gpd
import rasterio
from shapely.geometry import box
from IPython.display import Image, display

from lsdviztools3.io.vector import load_vector
from lsdviztools3.render.style import MapStyle
from lsdviztools3.render.hillshade import render_hillshade
from lsdviztools3.render.basins import render_basins
from lsdviztools3.render.channels import render_channels
from lsdviztools3.render.points import render_points
from lsdviztools3.render.base import save_figure

basins = load_vector("gador_chi_basin_polygons.fgb")
basins["area_km2"] = basins.geometry.area / 1e6

# Does any basin touch the edge of the DEM? (It should not.)
with rasterio.open("gador_DEM.tif") as src:
    dem_edge = box(*src.bounds).exterior.buffer(45)   # 1.5 pixels wide
basins["touches_dem_edge"] = basins.geometry.intersects(dem_edge)

# Requested and snapped outlets as point layers
requested = gpd.GeoDataFrame(outlets, geometry=gpd.points_from_xy(outlets.longitude, outlets.latitude),
                             crs="EPSG:4326")
snapped = pd.read_csv("gador_chi_basin_outlets.csv")
snapped = gpd.GeoDataFrame(snapped, geometry=gpd.points_from_xy(snapped.longitude, snapped.latitude),
                           crs="EPSG:4326")
basins["snap_distance_m"] = requested.to_crs(basins.crs).distance(snapped.to_crs(basins.crs)).round(1)

basins.drop(columns="geometry")

In [ ]:
style = MapStyle(figure_size="15c", transparent=False)

fig = render_hillshade("gador_hillshade.tif", shade=False, style=replace(style, cmap="gray"))
basins["label"] = basins["basin_key"].astype(int)
fig = render_basins(basins, label_col="label", fill="orange", transparency=50,
                    style=replace(style, line_width="0.8p", line_color="black"), fig=fig)
fig = render_points(requested, symbol="x", pen="1.5p,red",
                    style=replace(style, point_size="0.3c"), fig=fig)
fig = render_points(snapped, symbol="c", pen="0.5p,black",
                    style=replace(style, point_size="0.2c"), fig=fig)
save_figure(fig, "gador_basins.png", style=style)

display(Image("gador_basins.png", width=700))

## Figure 2: Channel long profiles

A **long profile** is elevation against distance upstream from the outlet. `lsdviztools3` does not have a long-profile function, so we use matplotlib on `gador_chi_data_maps.csv`, which has `flow_distance` (distance upstream from the basin outlet, in m) and `elevation` for every channel pixel.

Each channel pixel carries a `source_key`, the channel head it belongs to. The pixels at the outlet carry the key of the **longest channel** (the main stem), so we draw that channel as a black line and all other channels as grey dots.

In [ ]:
import matplotlib.pyplot as plt

chi_points = pd.read_csv("gador_chi_data_maps.csv")

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, (b, pts) in zip(axes.flat, chi_points.groupby("basin_key")):
    ax.scatter(pts["flow_distance"] / 1000, pts["elevation"], s=1, c="0.6", label="all channels")
    main_source = pts.loc[pts["flow_distance"].idxmax(), "source_key"]
    main = pts[pts["source_key"] == main_source].sort_values("flow_distance")
    ax.plot(main["flow_distance"] / 1000, main["elevation"], c="k", lw=1.2, label="longest channel")
    ax.set_title(f"Basin {b}")
    ax.set_xlabel("Distance upstream from outlet (km)")
    ax.set_ylabel("Elevation (m)")
    ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig("gador_long_profiles.png", dpi=150)
plt.show()

Look for **flat reaches**, for example at about 1850 m in basin 1. The Sierra de Gádor is karst (limestone and dolomite), and its high plateau has closed depressions with no surface outlet. Before routing flow, `lsdtt3` fills these depressions, so the channel crosses them as a flat line. Flat reaches like this are an artefact of filling, not real river reaches.

## Figure 3: Chi plots with fitted segments

We use two `lsdviztools3` chart functions (matplotlib):

* `load_chi_fit` reads the fit nodes and segments and gives every channel pixel the $k_{sn}$ of the segment it belongs to.
* `plot_chi_profile` draws one basin: the observed elevations as points coloured by $\log_{10} k_{sn}$ of their fitted segment, with the fitted straight-line segments as thin dark lines on top.

The colour scale is different in each panel. The stacked plot after this one uses one colour scale for all four basins.

In [ ]:
from lsdviztools3.io.chi import load_chi_fit
from lsdviztools3.charts import plot_chi_profile, plot_chi_stacked

chi_fit = load_chi_fit("gador_chi_fit_nodes_mn_0p450.csv", "gador_chi_fit_segments_mn_0p450.csv")

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, b in zip(axes.flat, sorted(chi_fit["basin_index"].unique())):
    plot_chi_profile(chi_fit, basin=int(b), ax=ax, cmap="viridis", point_size=3)
fig.tight_layout()
fig.savefig("gador_chi_profiles.png", dpi=150)
plt.show()

`plot_chi_stacked` puts all four basins side by side (each shifted along the chi axis) with one shared colour scale, so their steepness can be compared.

In [ ]:
fig, ax = plot_chi_stacked(chi_fit, cmap="viridis")
fig.tight_layout()
fig.savefig("gador_chi_stacked.png", dpi=150)
plt.show()

`lsdtt-chi-analysis` also writes a summary of $k_{sn}$ for each basin. The `main_stem_*` columns use only the main stem. $k_{sn}$ here has units of m$^{2\theta}$ = m$^{0.9}$.

In [ ]:
ksn_summary = pd.read_csv("gador_chi_ksn_basin_summary_mn_0p450.csv")
ksn_summary[["basin_index", "n_segments", "median_k_sn", "chi_weighted_mean_k_sn",
             "main_stem_median_k_sn"]].round(1)

## Figure 4: The channel network over the hillshade

First the whole channel network from Step 3, with each Strahler stream order drawn separately (bigger rivers get thicker, darker lines) using `render_channels`. The basin outlines are drawn on top as red lines (we turn the polygons into their boundary lines and draw those with `render_channels` too).

In [ ]:
lines = load_vector("gador_channel_network_lines.fgb")
print("Number of channel links in each stream order:")
print(lines["stream_order"].value_counts().sort_index())

fig = render_hillshade("gador_hillshade.tif", shade=False, style=replace(style, cmap="gray"))

colours = {1: "lightskyblue", 2: "deepskyblue", 3: "dodgerblue",
           4: "blue", 5: "navy", 6: "midnightblue", 7: "black"}
for order in sorted(lines["stream_order"].unique()):
    subset = lines[lines["stream_order"] == order]
    order_style = replace(style, line_width=f"{0.4 * order:.1f}p",
                          line_color=colours.get(int(order), "black"))
    fig = render_channels(subset, style=order_style, fig=fig)

outlines = basins.copy()
outlines["geometry"] = basins.boundary
fig = render_channels(outlines, style=replace(style, line_width="1.2p", line_color="red"), fig=fig)

save_figure(fig, "gador_channel_network.png", style=style)
display(Image("gador_channel_network.png", width=700))

Now the channels in our four basins, coloured by the $k_{sn}$ of their fitted chi segment (`color_by="k_sn"`, on a log scale). This is a map view of the colours in Figure 3. We leave out segments with $k_{sn} \le 0$ (flat or slightly uphill fits, mostly across filled depressions), which cannot be shown on a log scale.

In [ ]:
seg_lines = load_vector("gador_chi_fit_segment_lines_mn_0p450.fgb")

fig = render_hillshade("gador_hillshade.tif", shade=False, style=replace(style, cmap="gray"))
fig = render_channels(seg_lines[seg_lines["k_sn"] > 0], color_by="k_sn", log_color=True,
                      cmap="viridis", colorbar=True, colorbar_label="log@-10@- k@-sn@-",
                      style=replace(style, line_width="1.5p"), fig=fig)
save_figure(fig, "gador_ksn_map.png", style=style)
display(Image("gador_ksn_map.png", width=700))

## Things to try

* **Concavity.** Change `m_over_n` to 0.3 or 0.6, rerun Step 4 and the chi plots. (The output names change too: `mn_0p300`, `mn_0p600`.) For which value do the tributaries line up best with the main stem? `lsdtt-chi-analysis` can also test a range of values for you: remove the `m_over_n` line (it overrides the range) and set `mn_min`, `mn_max` and `mn_step`. If you also set `run_jackknife: true`, it writes a *disorder* statistic for each m/n to `gador_chi_disorder_sweep.csv`; the m/n with the lowest disorder is the one that best collapses the tributaries onto the main stem.
* **Segment fitting.** Set `fit_critical_divergence_metres` to 5 or 30. How does the number of segments change? What is noise in a 30 m DEM, and what is a real knickpoint?
* **Channel heads.** Change `threshold_contributing_pixels` to 100 or 1000 in Steps 3 and 4.
* **Compare the sides.** Are the south-draining basins (0 and 1) steeper than the north-draining ones (2 and 3)?